링크 : https://www.kaggle.com/competitions/the-nature-conservancy-fisheries-monitoring/overview

# 1. 주제: 어선에서 촬영한 물고기 사진을 분석하여 참치·상어 등 8개 어종 분류

# 2. 데이터
### (1) train.zip
- zipped folder of all train images. The train folders are organized by fish species labels
### (2) test_stg1.zip / test_stg2.zip
- zipped folder of all test images in stage 1 / in stage 2 (not available until the second stage of the competition)
### (3) sample_submission_stg1.csv / sample_submission_stg2.csv
- a sample submission file in the correct format

# 3. 데이터 흐름
- 핵심: 작은 CNN을 3개의 Fold에서 각각 학습시킨 다음, 3개 CNN의 테스트 예측 확률을 평균내서 최종 어종을 분류하는 코드
### (1) 이미지 데이터 전처리
- 학습·테스트 이미지를 불러와 32*32 크기로 resize하고, 픽셀 값을 0~1로 정규화
- 학습 데이터의 어종 라벨은 One-Hot Encoding으로 변환

### (2) CNN 모델 생성
- Convolution → Pooling → Dense Layer로 구성된 CNN 모델을 생성
- 마지막에는 Softmax를 사용해 8개 어종의 분류 확률을 출력

### (3) 3-Fold 교차검증으로 학습
- 전체 학습 데이터를 3개로 나누어 3개의 CNN 모델을 각각 학습
- 각 모델은 일부 데이터를 검증 데이터로 사용하고, EarlyStopping으로 과적합을 방지

### (4) 테스트 데이터 예측 및 앙상블
- 학습된 3개의 CNN 모델로 테스트 이미지를 각각 예측
- 각 모델이 출력한 8개 어종의 확률을 평균하여 최종 예측값을 만듦

### (5) 제출 파일 생성
최종 예측 확률과 이미지 파일명을 합쳐 Kaggle 제출용 CSV 파일을 생성

# 3-1. 주요 코드
### (1) CNN 모델 생성

In [ ]:
def create_model():
    model = Sequential()
    model.add(ZeroPadding2D((1, 1), input_shape=(3, 32, 32), dim_ordering='th'))
    model.add(Convolution2D(4, 3, 3, activation='relu', dim_ordering='th'))
    model.add(ZeroPadding2D((1, 1), dim_ordering='th'))
    model.add(Convolution2D(4, 3, 3, activation='relu', dim_ordering='th'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2), dim_ordering='th'))

    model.add(ZeroPadding2D((1, 1), dim_ordering='th'))
    model.add(Convolution2D(8, 3, 3, activation='relu', dim_ordering='th'))
    model.add(ZeroPadding2D((1, 1), dim_ordering='th'))
    model.add(Convolution2D(8, 3, 3, activation='relu', dim_ordering='th'))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2), dim_ordering='th'))

    model.add(Flatten())
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(8, activation='softmax'))

    sgd = SGD(lr=1e-2, decay=1e-6, momentum=0.9, nesterov=True)
    model.compile(optimizer=sgd, loss='categorical_crossentropy')

    return model

### (2) 3개 모델의 예측값 평균 도출

In [ ]:
yfull_train = dict()
    kf = KFold(len(train_id), n_folds=nfolds, shuffle=True, random_state=random_state)
    num_fold = 0
    sum_score = 0
    models = []
    for train_index, test_index in kf:
        model = create_model()
        X_train = train_data[train_index]
        Y_train = train_target[train_index]
        X_valid = train_data[test_index]
        Y_valid = train_target[test_index]

        num_fold += 1
        print('Start KFold number {} from {}'.format(num_fold, nfolds))
        print('Split train: ', len(X_train), len(Y_train))
        print('Split valid: ', len(X_valid), len(Y_valid))

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=3, verbose=0),
        ]
        model.fit(X_train, Y_train, batch_size=batch_size, nb_epoch=nb_epoch,
              shuffle=True, verbose=2, validation_data=(X_valid, Y_valid),
              callbacks=callbacks)

        predictions_valid = model.predict(X_valid.astype('float32'), batch_size=batch_size, verbose=2)
        score = log_loss(Y_valid, predictions_valid)
        print('Score log_loss: ', score)
        sum_score += score*len(test_index)

        # Store valid predictions
        for i in range(len(test_index)):
            yfull_train[test_index[i]] = predictions_valid[i]

        models.append(model)

    score = sum_score/len(train_data)
    print("Log_loss train independent avg: ", score)

# 4. 새롭게 알게 된 내용 / 어려운 내용 / 배울 점
- CNN을 활용한 이미지 분류: CNN을 이용하면 이미지의 특징을 자동으로 추출하여 물고기 종류를 분류할 수 있다는 것을 알게 되었다.
- 데이터 전처리와 교차검증의 중요성: 이미지 크기를 통일하고 픽셀값을 정규화하는 과정이 필요하며, 3-Fold 교차검증을 통해 모델의 성능을 확인하는 방법을 배웠다.
- 여러 모델의 결과를 결합하는 방법: 여러 CNN 모델의 예측 확률을 평균내어 최종 결과를 만드는 앙상블 방법을 알게 되었고, 여러 모델의 결과를 효과적으로 활용하는 것이 중요하다는 것을 배웠다.